In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from PIL import Image
from collections import Counter

In [ ]:
CLASSES    = ['NORMAL', 'PNEUMONIA']
CSV_PATH   = './dataset.csv'
INPUT_SIZE = (244, 244)

BATCH_SIZE = 32

In [ ]:
df= pd.read_csv(CSV_PATH)
df.info()

In [ ]:
df.head()

In [ ]:
#TODO: refactor code

# Data augmentation and transforms
train_transforms = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Split into train and val (since merged, we need to split)
label_indices = [0 if x=='NORMAL' else 1 for x in df['labels']]
train_paths, val_paths, train_labels, val_labels = train_test_split(df['images'], label_indices, test_size=0.2, stratify=label_indices, random_state=42)

# WeightedRandomSampler for training
class_counts_train = Counter(train_labels)
class_weights = [1.0 / class_counts_train[i] for i in range(len(CLASSES))]
sample_weights = [class_weights[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# Custom dataset class
class XRayDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Create datasets
train_dataset = XRayDataset(train_paths, train_labels, transform=train_transforms)
val_dataset = XRayDataset(val_paths, val_labels, transform=val_transforms)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Model: ResNet18
model = models.resnet18(weights='DEFAULT')
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Binary classification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

class_counts= df['labels'].value_counts()	#TODO: refactor this part
# Loss function: Weighted CrossEntropy for imbalance
pos_weight = class_counts['NORMAL'] / class_counts['PNEUMONIA']
weights = torch.tensor([1.0, pos_weight]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# LR Scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [ ]:
# Training function
def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    best_recall = 0.0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long()
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_dataset)
        
        # Validation
        model.eval()
        val_preds = []
        val_labels_list = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.numpy())
        
        from sklearn.metrics import recall_score
        recall = recall_score(val_labels_list, val_preds, pos_label=1)
        
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Val Recall: {recall:.4f}')
        
        if recall > best_recall:
            best_recall = recall
            torch.save(model.state_dict(), 'xray_classifier.pth')
    
    return model

trained_model = train_model(model, criterion, optimizer, scheduler, num_epochs=10)

Epoch 1/10, Loss: 0.1735, Val Recall: 0.5524
Epoch 2/10, Loss: 0.1191, Val Recall: 0.8398
Epoch 3/10, Loss: 0.1051, Val Recall: 0.9764
Epoch 4/10, Loss: 0.0997, Val Recall: 0.9588
Epoch 5/10, Loss: 0.0936, Val Recall: 0.9788
Epoch 6/10, Loss: 0.0826, Val Recall: 0.9611
Epoch 7/10, Loss: 0.0802, Val Recall: 0.9658
Epoch 8/10, Loss: 0.0540, Val Recall: 0.9694
Epoch 9/10, Loss: 0.0477, Val Recall: 0.9611
Epoch 10/10, Loss: 0.0423, Val Recall: 0.9741


## Save configuration

In [ ]:
config = {
    'model': 'ResNet18',
    'input_size': INPUT_SIZE,
    'classes': CLASSES,
    # 'best_threshold': best_threshold  #TODO: do we need this?
}

with open('xray_classifier.json', 'w') as f:
    import json
    json.dump(config, f)

print("Model and config saved.")